# AudioMNIST Training Setup

## Prerequisites

Before training your CNN, you need to preprocess the raw audio data into HDF5 format.

## Preprocessing

Run
```bash
python preprocess.py \
    --source ./data \
    --destination ./preprocessed_data \
    --meta ./data/audioMNIST_meta.txt
```

### Parameters:
- `--source` - Path to raw audio data directory
- `--destination` - Where to save preprocessed HDF5 files
- `--meta` - Path to metadata JSON file

---

**Note:** This preprocessing step only needs to be run once. The files will be reused for all subsequent training runs.


In [ ]:
from torch.utils.data import DataLoader
from torch.utils.data import Subset
from src_base.AudioMnist import AudioMNIST
from src_base.base_cnn import BaseCNN
import torch

# For digit classification with AudioNet
train_dataset = AudioMNIST(
    root='./preprocessed_data',
    model_type='AudioNet',
    task='digit',
    split=0,
    mode='train'
)

val_dataset = AudioMNIST(
    root='./preprocessed_data',
    model_type='AudioNet',
    task='digit',
    split=0,
    mode='validate'
)

test_dataset = AudioMNIST(
    root='./preprocessed_data',
    model_type='AudioNet',
    task='digit',
    split=0,
    mode='test'
)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Extract only digit labels (index 0)
def collate_fn(batch):
    data, labels = zip(*batch)
    data = torch.stack(list(data)).unsqueeze(1)
    digit_labels = torch.tensor([l.flatten()[0].item() for l in labels])
    return data, digit_labels

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, collate_fn=collate_fn)

# small set for testing code
train_subset = Subset(train_dataset, range(len(train_dataset) // 10))
val_subset = Subset(val_dataset, range(len(val_dataset) // 10))

__s_train_loader = DataLoader(train_subset, batch_size=64, shuffle=True, collate_fn=collate_fn)
__s_val_loader = DataLoader(val_subset, batch_size=64, shuffle=False, collate_fn=collate_fn)

# Initialize model
model = BaseCNN(input_size=8000, nb_class=10)

# Train
nb_epochs = 30
tr_loss,val_accur = model.train_mod(nb_epoch=nb_epochs, trainloader=__s_train_loader, val_dataset=__s_val_loader)

device: cuda:0
device: cuda:0
Starting training over 30


RuntimeError: 0D or 1D target tensor expected, multi-target not supported

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
fig, axs = plt.subplots(1, 2, figsize=(15, 5))
axs[0].plot(np.arange(nb_epochs), tr_loss, label="training loss")
axs[1].plot(np.arange(nb_epochs), np.array(val_accur) / 100, label="validation accuracy")

for ax in axs:
    ax.set_xlabel("Epoch")
    ax.legend()
plt.show()